# Review 3 — Complete Analysis: Custom CUDA Kernels vs. PyTorch
### Layers covered: ReLU, MatMul (2 versions), Softmax, LayerNorm, Conv2d, plus an end-to-end mini network

**Before running:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.

What's new compared with Review 2:
1. Three new layer kernels: **Softmax, LayerNorm, Conv2d**
2. A **faster matmul** (register blocking) next to the Review 2 tiled one
3. A fair benchmark across **all** layers, saved as a CSV table
4. A **summary graph** that answers the project question: where does hand-written CUDA hold up, and where doesn't it?
5. An **end-to-end mini network** (matmul -> ReLU -> matmul -> softmax) built from our own kernels

All outputs are saved to `results/`.

## Step 0 — Clean start and environment check

In [ ]:
# Clear any broken build cache from earlier attempts (fixes the "cannot open shared object file" error)
!rm -rf /root/.cache/torch_extensions
!nvcc --version
!nvidia-smi

In [ ]:
import torch, os, time
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn.functional as F

assert torch.cuda.is_available(), "No GPU -> Runtime > Change runtime type > GPU"
device = torch.device("cuda")
print("PyTorch:", torch.__version__, "| CUDA (torch build):", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

# Fair comparison: our kernels use plain FP32, so stop PyTorch from silently using TF32 Tensor Cores.
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
os.makedirs("results", exist_ok=True)

## Step 1 — All kernels in one file, compiled once

| Layer | How our kernel works | Type of workload |
|---|---|---|
| ReLU | one thread per element (grid-stride loop) | memory-bound |
| MatMul v1 | 16x16 shared-memory tiles (Review 2) | compute-bound |
| MatMul v2 | 64x64 tiles, each thread computes a 4x4 block in registers | compute-bound |
| Softmax | one thread-block per row, shared-memory max/sum reductions | memory + reduction |
| LayerNorm | one thread-block per row, mean/variance reductions | memory + reduction |
| Conv2d | one thread per output pixel, direct sliding window (no tricks) | compute-bound |

Compiling takes about 1-2 minutes the first time. **If it fails, expand the output and read the first `error:` line above the traceback** — that's the real cause.

In [ ]:
cuda_source = r'''
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <cfloat>

#define CHECK_INPUT(x) TORCH_CHECK((x).is_cuda() && (x).scalar_type() == torch::kFloat32, #x " must be a float32 CUDA tensor")
#define CHECK_LAUNCH() TORCH_CHECK(cudaGetLastError() == cudaSuccess, "CUDA kernel launch failed")

static int next_pow2(int x) { int p = 1; while (p < x) p <<= 1; return p; }

// ---------------------------------------------------------------- ReLU
__global__ void relu_kernel(const float* __restrict__ x, float* __restrict__ y, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int i = idx; i < n; i += stride) { float v = x[i]; y[i] = v > 0.0f ? v : 0.0f; }
}
torch::Tensor relu_forward(torch::Tensor x) {
    CHECK_INPUT(x);
    auto xc = x.contiguous();
    auto y = torch::empty_like(xc);
    int n = xc.numel();
    int threads = 256, blocks = (n + threads - 1) / threads;
    if (blocks > 65535) blocks = 65535;
    relu_kernel<<<blocks, threads>>>(xc.data_ptr<float>(), y.data_ptr<float>(), n);
    CHECK_LAUNCH();
    return y;
}

// ------------------------------------------- MatMul v1: shared-memory tiling
#define TILE 16
__global__ void matmul_tiled_kernel(const float* __restrict__ A, const float* __restrict__ B,
                                    float* __restrict__ C, int M, int K, int N) {
    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];
    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y * TILE + ty, col = blockIdx.x * TILE + tx;
    float acc = 0.0f;
    int numTiles = (K + TILE - 1) / TILE;
    for (int t = 0; t < numTiles; ++t) {
        int aCol = t * TILE + tx, bRow = t * TILE + ty;
        As[ty][tx] = (row < M && aCol < K) ? A[row * K + aCol] : 0.0f;
        Bs[ty][tx] = (bRow < K && col < N) ? B[bRow * N + col] : 0.0f;
        __syncthreads();
#pragma unroll
        for (int i = 0; i < TILE; ++i) acc += As[ty][i] * Bs[i][tx];
        __syncthreads();
    }
    if (row < M && col < N) C[row * N + col] = acc;
}
torch::Tensor matmul_forward(torch::Tensor A, torch::Tensor B) {
    CHECK_INPUT(A); CHECK_INPUT(B);
    auto Ac = A.contiguous(), Bc = B.contiguous();
    int M = Ac.size(0), K = Ac.size(1), N = Bc.size(1);
    TORCH_CHECK(Bc.size(0) == K, "inner dimensions must match");
    auto C = torch::empty({M, N}, Ac.options());
    dim3 block(TILE, TILE), grid((N + TILE - 1) / TILE, (M + TILE - 1) / TILE);
    matmul_tiled_kernel<<<grid, block>>>(Ac.data_ptr<float>(), Bc.data_ptr<float>(), C.data_ptr<float>(), M, K, N);
    CHECK_LAUNCH();
    return C;
}

// ------------------------- MatMul v2: shared tiling + register blocking (4x4 per thread)
#define BM 64
#define BN 64
#define BK 16
#define TM 4
#define TN 4
__global__ void matmul_reg_kernel(const float* __restrict__ A, const float* __restrict__ B,
                                  float* __restrict__ C, int M, int K, int N) {
    __shared__ float As[BK][BM];
    __shared__ float Bs[BK][BN];
    int tx = threadIdx.x, ty = threadIdx.y;       // 16 x 16 threads
    int tid = ty * 16 + tx;
    int rowBase = blockIdx.y * BM, colBase = blockIdx.x * BN;
    float acc[TM][TN];
#pragma unroll
    for (int i = 0; i < TM; ++i)
#pragma unroll
        for (int j = 0; j < TN; ++j) acc[i][j] = 0.0f;

    for (int k0 = 0; k0 < K; k0 += BK) {
#pragma unroll
        for (int i = 0; i < 4; ++i) {              // A tile: BM x BK = 1024 elements
            int idx = tid + i * 256, r = idx / BK, c = idx % BK;
            int gr = rowBase + r, gc = k0 + c;
            As[c][r] = (gr < M && gc < K) ? A[gr * K + gc] : 0.0f;
        }
#pragma unroll
        for (int i = 0; i < 4; ++i) {              // B tile: BK x BN = 1024 elements
            int idx = tid + i * 256, r = idx / BN, c = idx % BN;
            int gr = k0 + r, gc = colBase + c;
            Bs[r][c] = (gr < K && gc < N) ? B[gr * N + gc] : 0.0f;
        }
        __syncthreads();
#pragma unroll
        for (int k = 0; k < BK; ++k) {
            float a[TM], b[TN];
#pragma unroll
            for (int i = 0; i < TM; ++i) a[i] = As[k][ty * TM + i];
#pragma unroll
            for (int j = 0; j < TN; ++j) b[j] = Bs[k][tx * TN + j];
#pragma unroll
            for (int i = 0; i < TM; ++i)
#pragma unroll
                for (int j = 0; j < TN; ++j) acc[i][j] += a[i] * b[j];
        }
        __syncthreads();
    }
#pragma unroll
    for (int i = 0; i < TM; ++i)
#pragma unroll
        for (int j = 0; j < TN; ++j) {
            int r = rowBase + ty * TM + i, c = colBase + tx * TN + j;
            if (r < M && c < N) C[r * N + c] = acc[i][j];
        }
}
torch::Tensor matmul_reg_forward(torch::Tensor A, torch::Tensor B) {
    CHECK_INPUT(A); CHECK_INPUT(B);
    auto Ac = A.contiguous(), Bc = B.contiguous();
    int M = Ac.size(0), K = Ac.size(1), N = Bc.size(1);
    TORCH_CHECK(Bc.size(0) == K, "inner dimensions must match");
    auto C = torch::empty({M, N}, Ac.options());
    dim3 block(16, 16), grid((N + BN - 1) / BN, (M + BM - 1) / BM);
    matmul_reg_kernel<<<grid, block>>>(Ac.data_ptr<float>(), Bc.data_ptr<float>(), C.data_ptr<float>(), M, K, N);
    CHECK_LAUNCH();
    return C;
}

// ------------------------------------------------------- Softmax (row-wise)
__global__ void softmax_kernel(const float* __restrict__ X, float* __restrict__ Y, int rows, int cols) {
    extern __shared__ float sdata[];
    int row = blockIdx.x, tid = threadIdx.x;
    if (row >= rows) return;
    const float* xr = X + (size_t)row * cols;
    float* yr = Y + (size_t)row * cols;
    float m = -FLT_MAX;
    for (int i = tid; i < cols; i += blockDim.x) m = fmaxf(m, xr[i]);
    sdata[tid] = m; __syncthreads();
    for (int s = blockDim.x / 2; s > 0; s >>= 1) { if (tid < s) sdata[tid] = fmaxf(sdata[tid], sdata[tid + s]); __syncthreads(); }
    float row_max = sdata[0]; __syncthreads();
    float sum = 0.0f;
    for (int i = tid; i < cols; i += blockDim.x) { float e = expf(xr[i] - row_max); yr[i] = e; sum += e; }
    sdata[tid] = sum; __syncthreads();
    for (int s = blockDim.x / 2; s > 0; s >>= 1) { if (tid < s) sdata[tid] += sdata[tid + s]; __syncthreads(); }
    float row_sum = sdata[0];
    for (int i = tid; i < cols; i += blockDim.x) yr[i] = yr[i] / row_sum;
}
torch::Tensor softmax_forward(torch::Tensor x) {
    CHECK_INPUT(x);
    auto xc = x.contiguous();
    TORCH_CHECK(xc.dim() == 2, "softmax expects a 2D tensor [rows, cols]");
    int rows = xc.size(0), cols = xc.size(1);
    auto y = torch::empty_like(xc);
    int threads = next_pow2(cols < 1024 ? cols : 1024);
    if (threads < 32) threads = 32;
    softmax_kernel<<<rows, threads, threads * sizeof(float)>>>(xc.data_ptr<float>(), y.data_ptr<float>(), rows, cols);
    CHECK_LAUNCH();
    return y;
}

// ---------------------------------------------------------------- LayerNorm
__global__ void layernorm_kernel(const float* __restrict__ X, const float* __restrict__ gamma,
                                 const float* __restrict__ beta, float* __restrict__ Y,
                                 int rows, int cols, float eps) {
    extern __shared__ float sdata[];
    int row = blockIdx.x, tid = threadIdx.x;
    if (row >= rows) return;
    const float* xr = X + (size_t)row * cols;
    float* yr = Y + (size_t)row * cols;
    float s = 0.0f;
    for (int i = tid; i < cols; i += blockDim.x) s += xr[i];
    sdata[tid] = s; __syncthreads();
    for (int k = blockDim.x / 2; k > 0; k >>= 1) { if (tid < k) sdata[tid] += sdata[tid + k]; __syncthreads(); }
    float mean = sdata[0] / cols; __syncthreads();
    float v = 0.0f;
    for (int i = tid; i < cols; i += blockDim.x) { float d = xr[i] - mean; v += d * d; }
    sdata[tid] = v; __syncthreads();
    for (int k = blockDim.x / 2; k > 0; k >>= 1) { if (tid < k) sdata[tid] += sdata[tid + k]; __syncthreads(); }
    float inv_std = rsqrtf(sdata[0] / cols + eps);
    for (int i = tid; i < cols; i += blockDim.x) yr[i] = (xr[i] - mean) * inv_std * gamma[i] + beta[i];
}
torch::Tensor layernorm_forward(torch::Tensor x, torch::Tensor gamma, torch::Tensor beta, double eps) {
    CHECK_INPUT(x); CHECK_INPUT(gamma); CHECK_INPUT(beta);
    auto xc = x.contiguous(), gc = gamma.contiguous(), bc = beta.contiguous();
    TORCH_CHECK(xc.dim() == 2, "layernorm expects a 2D tensor [rows, cols]");
    int rows = xc.size(0), cols = xc.size(1);
    auto y = torch::empty_like(xc);
    int threads = next_pow2(cols < 1024 ? cols : 1024);
    if (threads < 32) threads = 32;
    layernorm_kernel<<<rows, threads, threads * sizeof(float)>>>(xc.data_ptr<float>(), gc.data_ptr<float>(),
                                                                 bc.data_ptr<float>(), y.data_ptr<float>(), rows, cols, (float)eps);
    CHECK_LAUNCH();
    return y;
}

// ------------------------------------------- Conv2d (direct, NCHW, naive)
__global__ void conv2d_kernel(const float* __restrict__ in, const float* __restrict__ w,
                              const float* __restrict__ bias, float* __restrict__ out,
                              int N, int Cin, int H, int W, int Cout, int KH, int KW,
                              int OH, int OW, int stride, int pad) {
    int ow = blockIdx.x * blockDim.x + threadIdx.x;
    int oh = blockIdx.y * blockDim.y + threadIdx.y;
    int nc = blockIdx.z, n = nc / Cout, co = nc % Cout;
    if (ow >= OW || oh >= OH || n >= N) return;
    float acc = 0.0f;
    for (int ci = 0; ci < Cin; ++ci)
        for (int kh = 0; kh < KH; ++kh) {
            int ih = oh * stride - pad + kh;
            if (ih < 0 || ih >= H) continue;
            for (int kw = 0; kw < KW; ++kw) {
                int iw = ow * stride - pad + kw;
                if (iw < 0 || iw >= W) continue;
                acc += in[((n * Cin + ci) * H + ih) * W + iw] * w[((co * Cin + ci) * KH + kh) * KW + kw];
            }
        }
    out[((n * Cout + co) * OH + oh) * OW + ow] = acc + bias[co];
}
torch::Tensor conv2d_forward(torch::Tensor input, torch::Tensor weight, torch::Tensor bias, int64_t stride, int64_t pad) {
    CHECK_INPUT(input); CHECK_INPUT(weight); CHECK_INPUT(bias);
    auto in = input.contiguous(), w = weight.contiguous(), b = bias.contiguous();
    int N = in.size(0), Cin = in.size(1), H = in.size(2), W = in.size(3);
    int Cout = w.size(0), KH = w.size(2), KW = w.size(3);
    TORCH_CHECK(w.size(1) == Cin, "channel mismatch");
    int OH = (H + 2 * pad - KH) / stride + 1, OW = (W + 2 * pad - KW) / stride + 1;
    auto out = torch::empty({N, Cout, OH, OW}, in.options());
    dim3 block(16, 16, 1), grid((OW + 15) / 16, (OH + 15) / 16, N * Cout);
    conv2d_kernel<<<grid, block>>>(in.data_ptr<float>(), w.data_ptr<float>(), b.data_ptr<float>(), out.data_ptr<float>(),
                                   N, Cin, H, W, Cout, KH, KW, OH, OW, (int)stride, (int)pad);
    CHECK_LAUNCH();
    return out;
}

'''

In [ ]:
from torch.utils.cpp_extension import load_inline

cpp_source = """
torch::Tensor relu_forward(torch::Tensor x);
torch::Tensor matmul_forward(torch::Tensor A, torch::Tensor B);
torch::Tensor matmul_reg_forward(torch::Tensor A, torch::Tensor B);
torch::Tensor softmax_forward(torch::Tensor x);
torch::Tensor layernorm_forward(torch::Tensor x, torch::Tensor gamma, torch::Tensor beta, double eps);
torch::Tensor conv2d_forward(torch::Tensor input, torch::Tensor weight, torch::Tensor bias, int64_t stride, int64_t pad);
"""

ext = load_inline(
    name="review3_ops",
    cpp_sources=cpp_source,
    cuda_sources=cuda_source,
    functions=["relu_forward", "matmul_forward", "matmul_reg_forward",
               "softmax_forward", "layernorm_forward", "conv2d_forward"],
    extra_cuda_cflags=["-O3"],
    verbose=True,
)
print("All kernels compiled successfully.")

## Step 2 — Correctness first
A speed number is meaningless if the answer is wrong, so every kernel is checked against PyTorch before any timing.
Reductions and convolutions add numbers in a different order than PyTorch does, so tiny floating-point differences are normal — we use a small tolerance.

In [ ]:
def check(name, mine, ref, atol=1e-4, rtol=1e-4):
    err = (mine - ref).abs().max().item()
    ok = torch.allclose(mine, ref, atol=atol, rtol=rtol)
    print(f"{'PASS' if ok else 'FAIL'}  {name:<22} max abs error = {err:.2e}")
    return ok

torch.manual_seed(0)
results_ok = []

x = torch.randn(1_000_000, device=device)
results_ok.append(check("ReLU", ext.relu_forward(x), torch.relu(x), atol=1e-6))

A = torch.randn(513, 300, device=device); B = torch.randn(300, 257, device=device)   # odd sizes on purpose
results_ok.append(check("MatMul v1 (tiled)", ext.matmul_forward(A, B), A @ B, atol=1e-3, rtol=1e-3))
results_ok.append(check("MatMul v2 (register)", ext.matmul_reg_forward(A, B), A @ B, atol=1e-3, rtol=1e-3))

s = torch.randn(256, 1000, device=device)       # 1000 is not a power of two, tests our reduction
results_ok.append(check("Softmax", ext.softmax_forward(s), F.softmax(s, dim=1), atol=1e-6))

ln = torch.randn(256, 768, device=device)
g = torch.randn(768, device=device); b = torch.randn(768, device=device)
results_ok.append(check("LayerNorm", ext.layernorm_forward(ln, g, b, 1e-5), F.layer_norm(ln, (768,), g, b, 1e-5), atol=1e-4))

xi = torch.randn(4, 8, 33, 33, device=device); wi = torch.randn(16, 8, 3, 3, device=device); bi = torch.randn(16, device=device)
results_ok.append(check("Conv2d", ext.conv2d_forward(xi, wi, bi, 1, 1), F.conv2d(xi, wi, bi, stride=1, padding=1), atol=1e-3, rtol=1e-3))

assert all(results_ok), "At least one kernel is wrong - fix before benchmarking"
print("\nAll kernels match PyTorch.")

## Step 3 — Benchmark every layer
Timing uses `torch.cuda.Event` (GPU-side clock), 10 warm-up runs, then the average of 30 runs.
We report **slowdown = custom time / PyTorch time** (1.0 = same speed, 5.0 = our kernel is 5x slower).

In [ ]:
def bench(fn, warmup=10, iters=30):
    for _ in range(warmup): fn()
    torch.cuda.synchronize()
    a, b = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    a.record()
    for _ in range(iters): fn()
    b.record(); torch.cuda.synchronize()
    return a.elapsed_time(b) / iters   # ms per call

rows = []
def add(op, cfg, custom_ms, torch_ms, **extra):
    rows.append(dict(op=op, config=cfg, custom_ms=custom_ms, torch_ms=torch_ms, slowdown=custom_ms / torch_ms, **extra))
    print(f"{op:<12} {cfg:<22} custom {custom_ms:9.4f} ms | torch {torch_ms:9.4f} ms | slowdown {custom_ms/torch_ms:6.2f}x")

# ReLU
for n in [1_000_000, 10_000_000, 50_000_000]:
    x = torch.randn(n, device=device)
    add("ReLU", f"n={n:,}", bench(lambda: ext.relu_forward(x)), bench(lambda: torch.relu(x)))

# Softmax
for r, cl in [(1024, 128), (1024, 1024), (4096, 1024), (4096, 4096)]:
    x = torch.randn(r, cl, device=device)
    add("Softmax", f"{r}x{cl}", bench(lambda: ext.softmax_forward(x)), bench(lambda: F.softmax(x, dim=1)))

# LayerNorm
for r, cl in [(1024, 256), (1024, 1024), (4096, 1024), (4096, 4096)]:
    x = torch.randn(r, cl, device=device); g = torch.ones(cl, device=device); b = torch.zeros(cl, device=device)
    add("LayerNorm", f"{r}x{cl}", bench(lambda: ext.layernorm_forward(x, g, b, 1e-5)),
        bench(lambda: F.layer_norm(x, (cl,), g, b, 1e-5)))

# Conv2d  (N, Cin, H, W, Cout), 3x3 kernel, pad 1
for N, Ci, H, Co in [(1, 3, 32, 16), (8, 16, 64, 32), (16, 64, 56, 64)]:
    x = torch.randn(N, Ci, H, H, device=device); w = torch.randn(Co, Ci, 3, 3, device=device); bb = torch.randn(Co, device=device)
    add("Conv2d", f"N{N} C{Ci}->{Co} {H}x{H}", bench(lambda: ext.conv2d_forward(x, w, bb, 1, 1), iters=10),
        bench(lambda: F.conv2d(x, w, bb, stride=1, padding=1), iters=10))

In [ ]:
# MatMul: three-way comparison (tiled v1, register v2, PyTorch/cuBLAS)
mm = []
print(f"{'N':>6} | {'v1 GFLOP/s':>11} | {'v2 GFLOP/s':>11} | {'torch GFLOP/s':>13}")
for n in [256, 512, 1024, 2048, 4096]:
    A = torch.randn(n, n, device=device); B = torch.randn(n, n, device=device)
    t1 = bench(lambda: ext.matmul_forward(A, B), iters=10)
    t2 = bench(lambda: ext.matmul_reg_forward(A, B), iters=10)
    tt = bench(lambda: A @ B, iters=10)
    fl = 2 * n**3
    g1, g2, gt = [fl / (t * 1e-3) / 1e9 for t in (t1, t2, tt)]
    mm.append((n, t1, t2, tt, g1, g2, gt))
    print(f"{n:>6} | {g1:>11.1f} | {g2:>11.1f} | {gt:>13.1f}")
add("MatMul v1", f"{mm[-1][0]}x{mm[-1][0]}", mm[-1][1], mm[-1][3])
add("MatMul v2", f"{mm[-1][0]}x{mm[-1][0]}", mm[-1][2], mm[-1][3])

In [ ]:
df = pd.DataFrame(rows)
df.to_csv("results/review3_benchmark.csv", index=False)
df.round(3)

## Step 4 — Graphs for the presentation

In [ ]:
# Graph 1: latency per layer, custom vs PyTorch
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, op in zip(axes.ravel(), ["ReLU", "Softmax", "LayerNorm", "Conv2d"]):
    d = df[df.op == op]
    xs = range(len(d)); w = 0.38
    ax.bar([i - w/2 for i in xs], d.custom_ms, w, label="Custom CUDA", color="#4C72B0")
    ax.bar([i + w/2 for i in xs], d.torch_ms, w, label="PyTorch", color="#F18F01")
    ax.set_xticks(list(xs)); ax.set_xticklabels(d.config, rotation=20, ha="right", fontsize=8)
    ax.set_yscale("log"); ax.set_ylabel("Latency (ms, log)"); ax.set_title(op); ax.legend(); ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig("results/r3_latency_by_layer.png", dpi=150); plt.show()

In [ ]:
# Graph 2: matmul throughput, three implementations
ns = [r[0] for r in mm]
plt.figure(figsize=(8, 5))
plt.plot(ns, [r[4] for r in mm], "o-", label="Custom v1: shared-memory tiling")
plt.plot(ns, [r[5] for r in mm], "s-", label="Custom v2: + register blocking")
plt.plot(ns, [r[6] for r in mm], "^-", label="PyTorch (cuBLAS)")
plt.xscale("log", base=2); plt.xlabel("Matrix size (N x N)"); plt.ylabel("Throughput (GFLOP/s)")
plt.title("MatMul: each optimization closes part of the gap"); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig("results/r3_matmul_optimization.png", dpi=150); plt.show()

In [ ]:
# Graph 3: the one-slide summary - largest size of each layer
last = df.groupby("op", sort=False).tail(1).reset_index(drop=True)
plt.figure(figsize=(8, 5))
colors = ["#2E8B57" if s < 1.5 else ("#F18F01" if s < 5 else "#C0392B") for s in last.slowdown]
plt.barh(last.op, last.slowdown, color=colors)
plt.axvline(1.0, color="black", linestyle="--", label="Same speed as PyTorch")
plt.xscale("log"); plt.xlabel("Slowdown vs PyTorch (custom time / PyTorch time, log scale)")
plt.title("Where does hand-written CUDA hold up?"); plt.legend(); plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout(); plt.savefig("results/r3_summary_slowdown.png", dpi=150); plt.show()

## Step 5 — End-to-end mini network (built only from our kernels)
`x -> matmul -> ReLU -> matmul -> softmax`, compared with the same network written in plain PyTorch.
This answers "can you show it working as a network?" — and checks that our kernels still agree with PyTorch when chained together.

In [ ]:
torch.manual_seed(1)
x  = torch.randn(256, 1024, device=device)
W1 = torch.randn(1024, 2048, device=device) * 0.03
W2 = torch.randn(2048, 512,  device=device) * 0.03

def net_custom(x): return ext.softmax_forward(ext.matmul_reg_forward(ext.relu_forward(ext.matmul_reg_forward(x, W1)), W2))
def net_torch(x):  return F.softmax(torch.relu(x @ W1) @ W2, dim=1)

check("Mini network output", net_custom(x), net_torch(x), atol=1e-4, rtol=1e-3)
t_c, t_t = bench(lambda: net_custom(x)), bench(lambda: net_torch(x))
print(f"Custom network: {t_c:.3f} ms | PyTorch network: {t_t:.3f} ms | slowdown {t_c/t_t:.2f}x")
print("Prediction rows sum to 1:", torch.allclose(net_custom(x).sum(1), torch.ones(256, device=device), atol=1e-4))

## How to explain your results (Review 3 discussion)

Fill in your own numbers from the table above, then use this structure:

- **Memory-bound layers (ReLU, Softmax, LayerNorm):** the GPU spends its time moving data, so a well-written kernel can land close to PyTorch. Any remaining gap comes from how efficiently the row reductions are done (PyTorch uses warp-level shuffles and vectorized loads; ours uses shared-memory trees).
- **Compute-bound layers (MatMul, Conv2d):** the gap is large because cuBLAS/cuDNN use register blocking, double buffering, Tensor Cores and (for conv) smarter algorithms such as implicit GEMM or Winograd. Our naive Conv2d uses none of these.
- **Optimization works:** MatMul v2 should sit clearly above v1 in Graph 2. That shows *why* each technique matters, even though cuBLAS stays ahead.
- **Fairness:** FP32 only, TF32 disabled, GPU-side timing, warm-up, and correctness checked before timing.

## Limitations and future work
- Forward pass only (no backward/gradients, so not trainable yet) and FP32 only
- Conv2d is a direct-convolution baseline; im2col + our matmul is the natural next optimization
- Could profile with NVIDIA Nsight Compute and compare against `torch.compile`